# NB04 - Matching Engine
### Cross-Benefit Duplicate Drug Billing Detection

**Schema:** `cross_benefit_audit`<br>
**Tables created:** `candidate_duplicate_pairs`<br>
**Depends on:** NB01, NB02, NB03

---

## What this notebook does

In the NCPDP D.0 and CMS-1500/UB-04 claim formats, the drug identifier differs between benefits - NDC on the pharmacy side, HCPCS on the medical side.

For Part B specialty drugs billed under J/Q codes, the CMS ASP crosswalk provides a tractable translation layer.

Production systems with commercial data typically use Generic Product Identifier (GPI) or RxNorm as the molecule identity layer; the CMS crosswalk is used here because it’s the public reference file that defines the target codes in scope.

This notebook runs **two passes**:

| Pass | Logic | What it misses |
|---|---|---|
| Naive | Join on `member_id + hcpcs_code` within date window | Any scenario where the two legs use different codes for the same molecule |
| Bridge | Join on `member_id + molecule` within date window, using the NB01 crosswalk as the translation layer | Less - but still generates false positives that need NB05 scoring |

The comparison is the core portfolio argument: **18 QCODE_GAP scenarios, 10 NEW_FORM_CROSS scenarios, and 8 DIFF_BIOSIM scenarios are invisible to the naive matcher.**

The molecule bridge is required to surface them.

### Output

| Table | Description |
|---|---|
| `candidate_duplicate_pairs` | All member-level rx/med pairs within the date window, scored, with ground-truth labels |

## 0 - Setup

In [0]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql import Window
import pandas as pd

spark = SparkSession.builder.getOrCreate()

RX_TABLE = "cross_benefit_audit.pharmacy_claims"
MED_TABLE = "cross_benefit_audit.medical_claims"
SEED_TABLE = "cross_benefit_audit.seeded_duplicate_events"
OUT_TABLE = "cross_benefit_audit.candidate_duplicate_pairs"

# Matching parameters
DATE_WINDOW_DAYS = 7 # abs(rx_date - med_date) <= this value
MIN_CONFIDENCE   = 0.40 # pairs below this threshold are not written to output

print("Setup complete.")
print(f"Date window : +/- {DATE_WINDOW_DAYS} days")
print(f"Min confidence threshold: {MIN_CONFIDENCE}")

Setup complete.
Date window : +/- 7 days
Min confidence threshold: 0.4


---
## 1 - Load tables

In [0]:
rx = spark.table(RX_TABLE)
med = spark.table(MED_TABLE)
seed = spark.table(SEED_TABLE)

print(f"pharmacy_claims  : {rx.count():,} rows")
print(f"medical_claims   : {med.count():,} rows")
print(f"seeded events    : {seed.count():,} rows")
print()
print("Seeded scenario counts:")
seed.groupBy("scenario_type").count().orderBy(F.desc("count")).show(truncate=False)

pharmacy_claims  : 552 rows
medical_claims   : 752 rows
seeded events    : 72 rows

Seeded scenario counts:
+---------------+-----+
|scenario_type  |count|
+---------------+-----+
|EXACT_SAME_CODE|20   |
|QCODE_GAP      |18   |
|NEAR_DATE      |16   |
|NEW_FORM_CROSS |10   |
|DIFF_BIOSIM    |8    |
+---------------+-----+



---
## 2 - Build the molecule bridge

In this project, the pharmacy claim carries an NDC; the medical claim carries a HCPCS code. The CMS ASP crosswalk from NB01 provides the two-sided translation:
<br><br>

```
pharmacy NDC  ──► crosswalk ──► **molecule** ◄── crosswalk ◄──  medical HCPCS
```
<br>
For this simulation, NB02 and NB03 pre-populated the `molecule` column on both tables. We use it here but restrict the naive pass to HCPCS-only joins to demonstrate exactly what a non-bridge engine misses.

In [0]:
# pharmacy side
rx_prep = (
    rx
    .select(
        F.col("claim_id").alias("rx_claim_id"),
        F.col("member_id"),
        F.col("service_date").alias("rx_date"),
        F.col("submitted_ndc").alias("rx_ndc"),
        F.col("hcpcs_code").alias("rx_hcpcs"),
        F.col("molecule").alias("rx_molecule"),
        F.col("drug_type").alias("rx_drug_type"),
        F.col("ingredient_cost").alias("rx_cost"),
        F.col("is_seeded_duplicate").alias("rx_is_seeded"),
        F.col("seed_event_id").alias("rx_seed_event_id"),
        F.col("seed_scenario_type").alias("rx_scenario"),
    )
)

# medical side
med_prep = (
    med
    .select(
        F.col("med_claim_id"),
        F.col("member_id"),
        F.col("service_date").alias("med_date"),
        F.col("hcpcs_code").alias("med_hcpcs"),
        F.col("molecule").alias("med_molecule"),
        F.col("drug_type").alias("med_drug_type"),
        F.col("hcpcs_units").alias("med_units"),
        F.col("allowed_amount").alias("med_allowed"),
        F.col("rendering_npi"),
        F.col("is_seeded_duplicate").alias("med_is_seeded"),
        F.col("seed_event_id").alias("med_seed_event_id"),
        F.col("seed_scenario_type").alias("med_scenario"),
    )
)

print("Pharmacy side:")
rx_prep.groupBy("rx_molecule", "rx_drug_type").count().orderBy("rx_molecule").show(20, truncate=False)

Pharmacy side:
+-------------+---------------+-----+
|rx_molecule  |rx_drug_type   |count|
+-------------+---------------+-----+
|aflibercept  |biosimilar     |24   |
|aflibercept  |originator     |54   |
|bevacizumab  |biosimilar     |52   |
|bevacizumab  |originator     |72   |
|infliximab   |biosimilar     |37   |
|infliximab   |originator     |48   |
|ocrelizumab  |originator     |56   |
|ocrelizumab  |new_formulation|26   |
|pembrolizumab|new_formulation|31   |
|pembrolizumab|originator     |67   |
|ranibizumab  |biosimilar     |47   |
|ranibizumab  |originator     |38   |
+-------------+---------------+-----+



---
## 3 - Pass 1: Naive HCPCS-to-HCPCS matcher

Join on `member_id + hcpcs_code` within the date window. This is the approach a system would use if it had no crosswalk and could not resolve molecule from code.

In [0]:
naive_matches = (
    rx_prep.alias("r")
    .join(
        med_prep.alias("m"),
        on=(
            (F.col("r.member_id") == F.col("m.member_id")) &
            (F.col("r.rx_hcpcs")  == F.col("m.med_hcpcs")) &
            (F.abs(F.datediff(F.col("m.med_date"), F.col("r.rx_date"))) <= DATE_WINDOW_DAYS)
        ),
        how="inner"
    )
    .select(
        F.col("r.rx_claim_id"),
        F.col("m.med_claim_id"),
        F.col("r.member_id"),
        F.col("r.rx_date"),
        F.col("m.med_date"),
        F.abs(F.datediff(F.col("m.med_date"), F.col("r.rx_date"))).alias("date_diff_days"),
        F.col("r.rx_hcpcs"),
        F.col("m.med_hcpcs"),
        F.col("r.rx_molecule"),
        F.lit("EXACT_CODE").alias("match_type"),
        F.col("r.rx_is_seeded"),
        F.col("r.rx_seed_event_id"),
        F.col("r.rx_scenario"),
        F.col("m.med_is_seeded"),
        F.col("m.med_seed_event_id"),
        F.col("m.med_scenario"),
    )
)

naive_count  = naive_matches.count()
naive_seeded = naive_matches.filter("rx_is_seeded = true AND med_is_seeded = true").count()

print(f"Naive HCPCS matcher - total candidate pairs : {naive_count:,}")
print(f"Naive HCPCS matcher - seeded pairs found    : {naive_seeded} / 72")
print()

# Which seeded scenarios does the naive matcher find?
print("Seeded scenarios found by naive matcher:")
(
    naive_matches
    .filter("rx_is_seeded = true AND med_is_seeded = true")
    .groupBy("rx_scenario")
    .count()
    .orderBy(F.desc("count"))
    .show(truncate=False)
)

Naive HCPCS matcher - total candidate pairs : 177
Naive HCPCS matcher - seeded pairs found    : 38 / 72

Seeded scenarios found by naive matcher:
+---------------+-----+
|rx_scenario    |count|
+---------------+-----+
|EXACT_SAME_CODE|22   |
|NEAR_DATE      |16   |
+---------------+-----+



---
## 4 - Pass 2-  Molecule bridge matcher

Join on `member_id + molecule` within the date window. Both legs translate to the same molecule regardless of which specific J or Q code was billed. This surfaces all five scenario types, including QCODE_GAP.

This is the engine the crosswalk from NB01 enables.

In [0]:
bridge_matches = (
    rx_prep.alias("r")
    .join(
        med_prep.alias("m"),
        on=(
            (F.col("r.member_id")    == F.col("m.member_id")) &
            (F.col("r.rx_molecule")  == F.col("m.med_molecule")) &
            (F.abs(F.datediff(F.col("m.med_date"), F.col("r.rx_date"))) <= DATE_WINDOW_DAYS)
        ),
        how="inner"
    )
    .select(
        F.col("r.rx_claim_id"),
        F.col("m.med_claim_id"),
        F.col("r.member_id"),
        F.col("r.rx_date"),
        F.col("m.med_date"),
        F.abs(F.datediff(F.col("m.med_date"), F.col("r.rx_date"))).alias("date_diff_days"),
        F.col("r.rx_hcpcs"),
        F.col("m.med_hcpcs"),
        F.col("r.rx_molecule"),
        # Classify by code alignment
        F.when(
            F.col("r.rx_hcpcs") == F.col("m.med_hcpcs"),
            F.lit("SAME_CODE")
        ).otherwise(
            F.lit("DIFF_CODE")
        ).alias("match_type"),
        F.col("r.rx_is_seeded"),
        F.col("r.rx_seed_event_id"),
        F.col("r.rx_scenario"),
        F.col("m.med_is_seeded"),
        F.col("m.med_seed_event_id"),
        F.col("m.med_scenario"),
    )
)

bridge_count  = bridge_matches.count()
bridge_seeded = bridge_matches.filter("rx_is_seeded = true AND med_is_seeded = true").count()

print(f"Molecule bridge matcher - total candidate pairs : {bridge_count:,}")
print(f"Molecule bridge matcher - seeded pairs found    : {bridge_seeded} / 72")
print()

print("Seeded scenarios found by molecule bridge:")
(
    bridge_matches
    .filter("rx_is_seeded = true AND med_is_seeded = true")
    .groupBy("rx_scenario", "match_type")
    .count()
    .orderBy("rx_scenario")
    .show(truncate=False)
)

Molecule bridge matcher - total candidate pairs : 405
Molecule bridge matcher - seeded pairs found    : 81 / 72

Seeded scenarios found by molecule bridge:
+---------------+----------+-----+
|rx_scenario    |match_type|count|
+---------------+----------+-----+
|DIFF_BIOSIM    |DIFF_CODE |8    |
|EXACT_SAME_CODE|SAME_CODE |22   |
|NEAR_DATE      |SAME_CODE |16   |
|NEAR_DATE      |DIFF_CODE |2    |
|NEW_FORM_CROSS |DIFF_CODE |13   |
|QCODE_GAP      |DIFF_CODE |20   |
+---------------+----------+-----+



### Key finding: scenarios invisible to the naive matcher

In [0]:
# Seeded pairs found by bridge but NOT by naive (gap)
naive_found_keys = naive_matches.filter("rx_is_seeded = true AND med_is_seeded = true").select(
    "rx_claim_id", "med_claim_id"
)

bridge_only = (
    bridge_matches
    .filter("rx_is_seeded = true AND med_is_seeded = true")
    .join(naive_found_keys, on=["rx_claim_id", "med_claim_id"], how="left_anti")
)

print(f"Seeded pairs found ONLY by molecule bridge (invisible to naive): {bridge_only.count()}")
print()
print("By scenario type:")
bridge_only.groupBy("rx_scenario", "rx_hcpcs", "med_hcpcs", "rx_molecule").count().orderBy("rx_scenario").show(30, truncate=False)

Seeded pairs found ONLY by molecule bridge (invisible to naive): 43

By scenario type:
+--------------+--------+---------+-------------+-----+
|rx_scenario   |rx_hcpcs|med_hcpcs|rx_molecule  |count|
+--------------+--------+---------+-------------+-----+
|DIFF_BIOSIM   |Q5118   |Q5107    |bevacizumab  |1    |
|DIFF_BIOSIM   |Q5107   |Q5118    |bevacizumab  |2    |
|DIFF_BIOSIM   |Q5104   |Q5103    |infliximab   |2    |
|DIFF_BIOSIM   |Q5128   |Q5124    |ranibizumab  |1    |
|DIFF_BIOSIM   |Q5124   |Q5128    |ranibizumab  |2    |
|NEAR_DATE     |Q5121   |Q5103    |infliximab   |1    |
|NEAR_DATE     |Q5124   |J2778    |ranibizumab  |1    |
|NEW_FORM_CROSS|J9277   |J9271    |pembrolizumab|10   |
|NEW_FORM_CROSS|J2351   |J2350    |ocrelizumab  |3    |
|QCODE_GAP     |Q5107   |J9035    |bevacizumab  |3    |
|QCODE_GAP     |Q5124   |J2778    |ranibizumab  |1    |
|QCODE_GAP     |Q5138   |J0178    |aflibercept  |3    |
|QCODE_GAP     |Q5128   |Q5124    |ranibizumab  |2    |
|QCODE_GAP     |Q

---
## 5 - Score candidate pairs

The confidence score has two components:

| Component | Weight | Rationale |
|---|---|---|
| Code alignment | 0-0.50 | Exact HCPCS match is stronger evidence than molecule-only match |
| Date proximity | 0-0.50 | Same-day overlap is stronger than 7-day gap |

**Code alignment score:**
- Same HCPCS on both legs: 0.50
- Different HCPCS, same molecule: 0.30

**Date proximity score:**
- 0 days: 0.50
- 1-3 days: 0.40
- 4-7 days: 0.25

In [0]:
def score_pairs(df):
    code_score = F.when(
        F.col("rx_hcpcs") == F.col("med_hcpcs"), 0.50
    ).otherwise(0.30)

    date_score = F.when(
        F.col("date_diff_days") == 0, 0.50
    ).when(
        F.col("date_diff_days") <= 3, 0.40
    ).otherwise(0.25)

    return df.withColumn(
        "confidence_score",
        F.round(code_score + date_score, 2)
    )

bridge_scored = score_pairs(bridge_matches)

print("Confidence score distribution (bridge matches):")
(
    bridge_scored
    .groupBy(F.round("confidence_score", 2).alias("score"))
    .count()
    .orderBy(F.desc("score"))
    .show()
)

Confidence score distribution (bridge matches):
+-----+-----+
|score|count|
+-----+-----+
|  1.0|   31|
|  0.9|   74|
|  0.8|   35|
| 0.75|   72|
|  0.7|   94|
| 0.55|   99|
+-----+-----+



---
## 6 - Attach ground-truth labels

A pair is a **true positive** when both the pharmacy and medical claim carry the same `seed_event_id`. That linkage was written by NB02 (pharmacy leg) and NB03 (medical leg) using the `seeded_duplicate_events` registry.

A pair where one or both legs are not seeded is either:
- A false positive (two legitimate claims that happen to overlap in date/molecule)
- A true negative that the matcher incorrectly surfaced

In [0]:
pairs_labeled = (
    bridge_scored
    .withColumn(
        "is_true_positive",
        (
            F.col("rx_is_seeded") &
            F.col("med_is_seeded") &
            (F.col("rx_seed_event_id") == F.col("med_seed_event_id"))
        )
    )
    .withColumn(
        "seed_event_id",
        F.when(F.col("is_true_positive"), F.col("rx_seed_event_id")).otherwise(F.lit(None))
    )
    .withColumn(
        "seed_scenario_type",
        F.when(F.col("is_true_positive"), F.col("rx_scenario")).otherwise(F.lit(None))
    )
    # Drop the per-leg seeded flags - the labeled columns replace them
    .drop(
        "rx_is_seeded", "rx_seed_event_id", "rx_scenario",
        "med_is_seeded", "med_seed_event_id", "med_scenario",
    )
    .filter(F.col("confidence_score") >= MIN_CONFIDENCE)
)

tp = pairs_labeled.filter("is_true_positive = true").count()
fp = pairs_labeled.filter("is_true_positive = false").count()

print(f"Pairs above confidence threshold ({MIN_CONFIDENCE}):")
print(f"  True positives  : {tp}")
print(f"  False positives : {fp}")
print(f"  Precision       : {tp / (tp + fp):.1%}")

Pairs above confidence threshold (0.4):
  True positives  : 72
  False positives : 333
  Precision       : 17.8%


In [0]:
# False negatives - seeded events not recovered at all
recovered_events = (
    pairs_labeled
    .filter("is_true_positive = true")
    .select("seed_event_id")
    .distinct()
)
all_event_ids = seed.select("seed_event_id").distinct()
missed_events = all_event_ids.join(recovered_events, "seed_event_id", "left_anti")

fn = missed_events.count()
print(f"\n  False negatives (seeded events not found) : {fn}")
print(f"  Recall                                     : {tp / (tp + fn):.1%}")


  False negatives (seeded events not found) : 0
  Recall                                     : 100.0%


---
## 7 - Write candidate duplicate pairs

In [0]:
OUTPUT_COLS = [
    "rx_claim_id", "med_claim_id", "member_id",
    "rx_date", "med_date", "date_diff_days",
    "rx_hcpcs", "med_hcpcs", "rx_molecule",
    "match_type", "confidence_score",
    "is_true_positive", "seed_event_id", "seed_scenario_type",
]

pairs_final = pairs_labeled.select(OUTPUT_COLS)

(
    pairs_final.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(OUT_TABLE)
)

print(f"Written: {OUT_TABLE}")
print(f"Row count: {spark.table(OUT_TABLE).count():,}")

Written: cross_benefit_audit.candidate_duplicate_pairs
Row count: 405


In [0]:
spark.table(OUT_TABLE).printSchema()

root
 |-- rx_claim_id: string (nullable = true)
 |-- med_claim_id: string (nullable = true)
 |-- member_id: string (nullable = true)
 |-- rx_date: date (nullable = true)
 |-- med_date: date (nullable = true)
 |-- date_diff_days: integer (nullable = true)
 |-- rx_hcpcs: string (nullable = true)
 |-- med_hcpcs: string (nullable = true)
 |-- rx_molecule: string (nullable = true)
 |-- match_type: string (nullable = true)
 |-- confidence_score: double (nullable = true)
 |-- is_true_positive: boolean (nullable = true)
 |-- seed_event_id: string (nullable = true)
 |-- seed_scenario_type: string (nullable = true)



---
## 8 - Detection analysis

### 8.1 - Naive vs bridge: head-to-head

In [0]:
print("=" * 65)
print("Naive HCPCS matcher vs Molecule bridge matcher")
print("=" * 65)
print(f"{'Metric':<40} {'Naive':>10} {'Bridge':>10}")
print("-" * 65)

# Naive stats
naive_tp = naive_matches.filter("rx_is_seeded = true AND med_is_seeded = true AND rx_seed_event_id = med_seed_event_id").count()
naive_fp = naive_matches.filter(
    ~(F.col("rx_is_seeded") & F.col("med_is_seeded") & (F.col("rx_seed_event_id") == F.col("med_seed_event_id")))
).count()
naive_fn = 72 - naive_tp
naive_prec = naive_tp / max(naive_tp + naive_fp, 1)
naive_rec  = naive_tp / 72

print(f"{'Candidate pairs generated':<40} {naive_count:>10,} {bridge_count:>10,}")
print(f"{'True positives (TP)':<40} {naive_tp:>10} {tp:>10}")
print(f"{'False positives (FP)':<40} {naive_fp:>10} {fp:>10}")
print(f"{'False negatives (FN - seeded not found)':<40} {naive_fn:>10} {fn:>10}")
print(f"{'Precision':<40} {naive_prec:>10.1%} {tp/(tp+fp):.1%}")
print(f"{'Recall':<40} {naive_rec:>10.1%} {tp/(tp+fn):.1%}")
print("=" * 65)

Naive HCPCS matcher vs Molecule bridge matcher
Metric                                        Naive     Bridge
-----------------------------------------------------------------
Candidate pairs generated                       177        405
True positives (TP)                              36         72
False positives (FP)                            141        333
False negatives (FN - seeded not found)          36          0
Precision                                     20.3% 17.8%
Recall                                        50.0% 100.0%


### 8.2 - Bridge - true positive breakdown by scenario

In [0]:
print("Bridge matcher - TP breakdown by seeded scenario:")
(
    spark.table(OUT_TABLE)
    .filter("is_true_positive = true")
    .groupBy("seed_scenario_type", "match_type")
    .agg(
        F.count("rx_claim_id").alias("tp_count"),
        F.round(F.avg("confidence_score"), 2).alias("avg_confidence"),
        F.round(F.avg("date_diff_days"), 1).alias("avg_date_diff"),
    )
    .orderBy("seed_scenario_type")
    .show(truncate=False)
)

Bridge matcher - TP breakdown by seeded scenario:
+------------------+----------+--------+--------------+-------------+
|seed_scenario_type|match_type|tp_count|avg_confidence|avg_date_diff|
+------------------+----------+--------+--------------+-------------+
|DIFF_BIOSIM       |DIFF_CODE |8       |0.8           |0.0          |
|EXACT_SAME_CODE   |SAME_CODE |20      |1.0           |0.0          |
|NEAR_DATE         |SAME_CODE |16      |0.9           |2.0          |
|NEW_FORM_CROSS    |DIFF_CODE |10      |0.73          |0.9          |
|QCODE_GAP         |DIFF_CODE |18      |0.76          |0.4          |
+------------------+----------+--------+--------------+-------------+



### 8.3 - False positive anatomy

False positives are legitimate claims where:
- The same member happened to have a pharmacy fill AND a medical infusion for the same
  molecule within the date window
- Both were clean (not seeded), but the molecule + date overlap triggers a candidate hit

These are the pairs NB05 must score down using dose quantity, unit reconciliation, provider co-location, and specialty pharmacy flags.

In [0]:
print("False positives by molecule and match type:")
(
    spark.table(OUT_TABLE)
    .filter("is_true_positive = false")
    .groupBy("rx_molecule", "match_type")
    .agg(
        F.count("rx_claim_id").alias("fp_count"),
        F.round(F.avg("confidence_score"), 2).alias("avg_confidence"),
        F.round(F.avg("date_diff_days"), 1).alias("avg_date_diff"),
    )
    .orderBy(F.desc("fp_count"))
    .show(20, truncate=False)
)

False positives by molecule and match type:
+-------------+----------+--------+--------------+-------------+
|rx_molecule  |match_type|fp_count|avg_confidence|avg_date_diff|
+-------------+----------+--------+--------------+-------------+
|bevacizumab  |DIFF_CODE |64      |0.63          |3.7          |
|bevacizumab  |SAME_CODE |43      |0.81          |4.2          |
|ranibizumab  |DIFF_CODE |37      |0.63          |3.8          |
|pembrolizumab|DIFF_CODE |30      |0.64          |3.5          |
|aflibercept  |SAME_CODE |29      |0.83          |3.8          |
|pembrolizumab|SAME_CODE |27      |0.84          |3.0          |
|infliximab   |DIFF_CODE |23      |0.64          |3.7          |
|ocrelizumab  |DIFF_CODE |19      |0.63          |3.8          |
|ocrelizumab  |SAME_CODE |19      |0.83          |3.5          |
|aflibercept  |DIFF_CODE |19      |0.6           |3.8          |
|ranibizumab  |SAME_CODE |13      |0.86          |3.3          |
|infliximab   |SAME_CODE |10      |0.82       

### 8.4 - QCODE_GAP - molecule bridge is required

In [0]:
print("QCODE_GAP pairs as seen by the molecule bridge:")
(
    spark.table(OUT_TABLE)
    .filter("seed_scenario_type = 'QCODE_GAP'")
    .select(
        "seed_event_id", "rx_hcpcs", "med_hcpcs", "rx_molecule",
        "date_diff_days", "confidence_score", "match_type"
    )
    .orderBy("rx_molecule", "seed_event_id")
    .show(20, truncate=False)
)

print()
print("These 18 pairs all have match_type = DIFF_CODE.")
print("A naive HCPCS-to-HCPCS matcher produces zero rows for any of them.")

QCODE_GAP pairs as seen by the molecule bridge:
+-------------+--------+---------+-----------+--------------+----------------+----------+
|seed_event_id|rx_hcpcs|med_hcpcs|rx_molecule|date_diff_days|confidence_score|match_type|
+-------------+--------+---------+-----------+--------------+----------------+----------+
|EVT-AFLIB-039|Q5138   |J0178    |aflibercept|0             |0.8             |DIFF_CODE |
|EVT-AFLIB-044|Q5138   |J0178    |aflibercept|1             |0.7             |DIFF_CODE |
|EVT-AFLIB-048|Q5138   |J0178    |aflibercept|0             |0.8             |DIFF_CODE |
|EVT-BEVAC-040|Q5107   |J9035    |bevacizumab|1             |0.7             |DIFF_CODE |
|EVT-BEVAC-041|Q5118   |J9035    |bevacizumab|0             |0.8             |DIFF_CODE |
|EVT-BEVAC-043|Q5107   |J9035    |bevacizumab|0             |0.8             |DIFF_CODE |
|EVT-BEVAC-050|Q5107   |J9035    |bevacizumab|1             |0.7             |DIFF_CODE |
|EVT-BEVAC-051|Q5118   |J9035    |bevacizumab|0     

---
## 9 - Summary

### Table written

| Table | Rows | Key fields |
|---|---|---|
| `candidate_duplicate_pairs` | varies | rx_claim_id, med_claim_id, molecule, confidence_score, is_true_positive |

### Detection results (molecule bridge, date window = 7 days)

| Scenario | Count seeded | Found | Notes |
|---|---|---|---|
| EXACT_SAME_CODE | 20 | 20 | High confidence (0.80+) |
| NEAR_DATE | 16 | 16 | Confidence scales with date proximity |
| QCODE_GAP | 18 | 18 | DIFF_CODE match - invisible to naive matcher |
| NEW_FORM_CROSS | 10 | 10 | DIFF_CODE match - invisible to naive matcher |
| DIFF_BIOSIM | 8 | 8 | DIFF_CODE match - invisible to naive matcher |

### Why false positives exist

186 of 200 members have claims on both benefits. Some coincidentally have pharmacy fills and medical infusions for the same molecule within 7 days without any duplicate billing. The molecule bridge flags them as candidates.

NB05 reduces false positives by adding:
- Dose quantity reconciliation (BILLUNITSPKG vs hcpcs_units)
- Specialty pharmacy flag (buy-and-bill drugs should not appear at a specialty pharmacy)
- Provider co-location (same rendering NPI on both legs is a strong signal)
- Claim cost comparison (two claims for same drug on same date should not both carry full cost)

---